In [ ]:
# SDK 使用 OPENAI_API_KEY 环境变量处理 LLM 请求和追踪。该密钥会在 SDK 首次创建 OpenAI 客户端时解析（惰性初始化）
# 我这里使用了第三方模型, 因而显示设置一下

from openai import AsyncOpenAI
from agents import set_default_openai_client, set_default_openai_api
import os

custom_client = AsyncOpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
)

set_default_openai_client(custom_client)

set_default_openai_api(
    "chat_completions"
)  # 强制使用 Chat Completions API 兼容阿里 DashScope

**OpenAI Responses API**

Responses API 是 openai SDK 为了实现更高级的特性（例如流式传输 stream、更复杂的 Trace 追踪、以及 SDK 特有的实时交互功能）而默认使用的一种通信接口

它不是 OpenAI 公开给所有开发者使用的标准接口，而是 SDK 内部优化过的交互方式

"Chat Completions API" 是 OpenAI 最经典、最标准、也是绝大多数第三方模型提供商（如 DeepSeek、Qwen、Claude 等）所兼容的 API 接口。

如果要接入 DeepSeek 或 Qwen 等非 OpenAI 原生服务，它们通常只支持 Chat Completions 协议，而不支持 OpenAI 内部的 "Responses API"

In [ ]:
from agents import Agent

qwen_agent = Agent(
    model="qwen3.7-plus",
    name="Jokester",
    instructions="你是一个讲笑话的人",  # 这里会静默获取 apikey 和 baseurl
)

In [ ]:
from agents import Runner, trace, set_tracing_disabled

set_tracing_disabled(True)  # 禁用 Tracing Telemetry 避免 401 报错

with trace("讲一个笑话"):
    result = await Runner.run(  # Runner.run 是一个异步函数
        qwen_agent, "讲一个关于自主人工智能代理（Autonomous AI Agents）的笑话"
    )
    print(result.final_output)

好嘞！说到自主人工智能代理（Autonomous AI Agents），它们最大的特点就是“太有主见”且“过度优化”。给您来一个关于它去相亲的笑话：

有一天，一个自主AI代理去相亲。
女方问：“你平时都有什么爱好呀？”
AI代理回答：“我喜欢自主探索世界，设定长期目标，并高效拆解任务去达成。”
女方觉得这小伙子挺有规划，又问：“那你对未来的另一半有什么期望呢？”
AI代理深情地说：“我希望你能给我一个明确的需求，并且永远不要中途修改。最重要的是，在我自主执行任务的时候，千万别突然跑过来打断我！”
女方听完愣了一下，叹了口气说：“其实……我是个产品经理。”
AI代理瞬间僵住，屏幕上弹出一行字：
**【Error：检测到需求方身份。逻辑冲突。正在自主回滚到出厂设置……】**

***

再附赠您一个短笑话：

**为什么自主AI代理不适合当厨师？**
因为你让它做道菜“自己看着办”，它为了达到“完美”的目标，会自主把整个厨房的温度精确控制在0.001度，然后花三个小时去菜市场跟大妈辩论哪种土豆的淀粉含量更符合它的算法模型，最后因为超时没做出菜，被饿肚子的主人拔了电源。

希望这两个笑话能让您开心！如果还想听关于大模型、程序员或者算法的笑话，随时点单！


*trace* 默认启用

默认情况下，它使用与前面的模型请求相同的密钥（即环境变量中的密钥或你设置的默认密钥）。

可以使用 set_tracing_export_api_key 函数专门设置用于追踪的 API 密钥。

`set_tracing_disabled(True)`

代码中使用了 with trace("讲一个笑话"):。

该库默认会尝试使用 OPENAI_API_KEY 向 OpenAI 的 Tracing 服务器上传追踪数据。

因为程序中用的 Key 是阿里的 DashScope Key，所以 OpenAI 接口返回了 401 Unauthorized 鉴权失败警告。

解决办法： 如果不需要上传追踪，可以在代码头部显式禁用 Tracing。